# AI工学101 — 第10回

## NumPy総合演習①：前処理パイプラインを作る

ここまでの9回で、NumPyの基礎は一通り学びました。

* 配列
* ベクトル演算
* ブロードキャスト
* Boolean Mask
* shape / reshape / axis
* 行列積
* 統計量
* 欠損値処理
* シャッフル
* EDA

今回は、それらを**1本の処理として組み立てる**演習です。

実際のAI開発では、

```text
CSV
 ↓
前処理
 ↓
特徴量作成
 ↓
学習
```

という流れになります。

今日はまだCSVは使わず、NumPyだけで「前処理パイプライン」を作ります。

---

# 今日のゴール

最終的に

```python
X_train
X_test
```

という機械学習に入力できるデータを作ります。

---

# 📖 講義（20分）

## パイプラインとは？

例えば料理。

```
材料を買う

↓

洗う

↓

切る

↓

焼く

↓

完成
```

これも一種のパイプラインです。

AIも同じ。

```
データ取得

↓

欠損値補完

↓

標準化

↓

シャッフル

↓

train/test分割

↓

学習
```

毎回この流れになります。

だからAIエンジニアは

**前処理を書く時間が非常に長い**

です。

---

# 今日使うデータ

```python
import numpy as np

X = np.array([
    [25,160,55],
    [30,170,65],
    [35,np.nan,70],
    [40,180,75],
    [45,190,300],
    [50,175,80],
    [55,np.nan,85],
    [60,185,90]
], dtype=float)
```

列は

```
年齢

身長

体重
```

です。

---

# 💻 実習1：データ概要

まず確認。

```python
print(X.shape)

print(X.dtype)

print(X)
```

AIでは

最初にshapeを見る癖をつけます。

---

# 💻 実習2：欠損確認

```python
print(np.isnan(X).sum(axis=0))
```

何個欠損があるか確認。

続いて平均。

```python
means = np.nanmean(
    X,
    axis=0
)

print(means)
```

---

# 💻 実習3：平均値補完

コピー。

```python
X_clean = X.copy()
```

欠損位置。

```python
rows, cols = np.where(
    np.isnan(X_clean)
)
```

補完。

```python
X_clean[rows, cols] = means[cols]
```

確認。

```python
print(X_clean)
```

---

# 💻 実習4：EDA

平均。

```python
print(
    X_clean.mean(axis=0)
)
```

標準偏差。

```python
print(
    X_clean.std(axis=0)
)
```

最小。

```python
print(
    X_clean.min(axis=0)
)
```

最大。

```python
print(
    X_clean.max(axis=0)
)
```

ここで

```
体重300kg
```

がかなり目立ちます。

実務なら

「入力ミス？」

「単位違い？」

などを確認します。

今日はそのまま残します。

---

# 💻 実習5：標準化

```python
mean = X_clean.mean(axis=0)

std = X_clean.std(axis=0)

X_scaled = (
    X_clean - mean
) / std
```

確認。

```python
print(
    X_scaled.mean(axis=0)
)

print(
    X_scaled.std(axis=0)
)
```

期待される結果。

```
平均≈0

標準偏差≈1
```

---

# 💻 実習6：シャッフル

再現性確保。

```python
np.random.seed(42)
```

インデックス。

```python
idx = np.random.permutation(
    len(X_scaled)
)
```

並び替え。

```python
X_scaled = X_scaled[idx]
```

確認。

```python
print(X_scaled)
```

---

# 💻 実習7：train/test分割

8件なので

```
6件

↓

2件
```

くらいにします。

```python
split = 6

X_train = X_scaled[:split]

X_test = X_scaled[split:]
```

確認。

```python
print(X_train.shape)

print(X_test.shape)
```

期待。

```
(6,3)

(2,3)
```

---

# 💻 実習8：前処理を関数にする

ここから一歩レベルアップ。

```python
def preprocess(X):

    X = X.copy()

    means = np.nanmean(
        X,
        axis=0
    )

    rows, cols = np.where(
        np.isnan(X)
    )

    X[rows, cols] = means[cols]

    mean = X.mean(axis=0)

    std = X.std(axis=0)

    X = (X - mean) / std

    return X
```

使う。

```python
X_processed = preprocess(X)

print(X_processed)
```

ここで初めて

「前処理を再利用できるコード」

になります。

---

# 📖 なぜ関数化するの？

今後

```
学習データ

↓

評価データ

↓

新しいデータ
```

全部に

同じ前処理

を適用します。

そのたびに

コピペすると

バグになります。

だから関数化します。

---

# ✍️ 演習

今日の課題。

```python
X = np.array([
    [20,150,50],
    [25,160,np.nan],
    [30,170,65],
    [35,180,70],
    [40,np.nan,75],
    [45,190,80],
    [50,200,85],
    [55,210,90]
], dtype=float)
```

---

## 問1

shapeを確認してください。

---

## 問2

欠損値を平均値で補完してください。

---

## 問3

標準化してください。

---

## 問4

seedを42に設定し、

シャッフルしてください。

---

## 問5

75%をtrain、

25%をtest

に分割してください。

---

## 問6（ボス戦👾）

今日作った

```python
preprocess()
```

を

自分で一から書いてください。

次の条件を満たします。

* コピーを作る
* 平均値補完
* 標準化
* 処理済み配列を返す

---

# 🌿 今日のまとめ

今日は、これまで学んだNumPyの知識を一つの流れとして組み立てました。

```text
NumPy配列
      ↓
欠損値確認
      ↓
平均値補完
      ↓
EDA
      ↓
標準化
      ↓
シャッフル
      ↓
train/test分割
```

これが、機械学習プロジェクトで繰り返し登場する**前処理パイプライン**です。

ここまでで「NumPyを使って機械学習用データを準備する」基礎体力はかなり身についてきました。

---

# 🔜 第11回予告

次回からはいよいよ **scikit-learn 編**に入ります。

最初のテーマは、

> **「NumPyで作ってきた前処理を、scikit-learnではどう書くのか？」**

です。

これまで自分で実装してきた

* 標準化
* 学習データ・テストデータ分割
* 前処理の流れ

を、`StandardScaler` や `train_test_split` を使って置き換えながら、「ライブラリが何をしているのか」を理解していきます。

ここからは、「NumPyで理解した中身」と「実務で使うライブラリ」が一本につながるフェーズに入ります。
